In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('my_data.csv')

In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [5]:
print(f"Initial Shape: {df.shape}")
print("Missing Values per Column:")
print(df.isnull().sum())

Initial Shape: (25907, 5)
Missing Values per Column:
timestamp     0
PIR_Motion    0
Accel_X       0
Accel_Y       0
Accel_Z       0
dtype: int64


## PIR must be 1 or 0

In [6]:
df = df[df["PIR_Motion"].isin([0, 1])]

## accelerometer in range of +/- 16g

In [7]:
accel_cols = ["Accel_X", "Accel_Y", "Accel_Z"]
for col in accel_cols:
    df = df[(df[col] >= -16.0) & (df[col] <= 16.0)]

## drop row if timestamp duplicates

In [8]:
df = df.drop_duplicates(subset=["timestamp"])

## transfer timestamp to sec

In [9]:
start_time = df["timestamp"].min()
df["elapsed_seconds"] = (
    df["timestamp"] - start_time
).dt.total_seconds().round(3)

## Calculate Acceleration Magnitude

In [10]:
df["Accel_Mag"] = np.sqrt(
    df["Accel_X"] ** 2 + df["Accel_Y"] ** 2 + (df["Accel_Z"] - 1.0) ** 2
)

## Moving Avg

In [12]:
df["Accel_Mag_Smooth"] = df["Accel_Mag"].rolling(window=5, min_periods=1).mean()
for col in accel_cols:
    df[f"{col}_Smooth"] = df[col].rolling(window=5, min_periods=1).mean()

In [13]:
movement_threshold = 0.3
df["Force_Detected"] = df["Accel_Mag_Smooth"] > movement_threshold

## Fused_Alarm_Trigger = when Alarm trigger 

In [14]:
df["Fused_Alarm_Trigger"] = (df["PIR_Motion"] == 1) & df["Force_Detected"]

## will not save data older than 48hr

In [16]:
retention_limit = df["timestamp"].max() - pd.Timedelta(hours=48)
df_processed = df[df["timestamp"] >= retention_limit].copy()

In [17]:
print(f"Final Cleaned Shape: {df_processed.shape}")
print(
    f"Data Loss / Outliers Removed: {len(df) - len(df_processed)}"
    f" rows ({((len(df) - len(df_processed)) / len(df)) * 100:.2f}%)"
)
print(f"Total Fused Alarm Events Detected: {df_processed['Fused_Alarm_Trigger'].sum()}")

Final Cleaned Shape: (25895, 13)
Data Loss / Outliers Removed: 0 rows (0.00%)
Total Fused Alarm Events Detected: 54


In [18]:
df_processed.to_csv("processed_data.csv", index=False)
print("Saved cleaned dataset to 'processed_data.csv'")

Saved cleaned dataset to 'processed_data.csv'
